# Variance Engine Check — sign-aware favorability & decomposition

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import run_pipeline as rp
t = rp.load(); out = rp.run(write=False)
print('tables:', list(t.keys()))

tables: ['dim_date', 'dim_department', 'dim_account', 'fact_actuals', 'fact_budget', 'fact_forecast', 'fact_headcount', 'fact_saas_metrics', 'fact_saas_metrics_budget', 'fact_saas_metrics_forecast']


In [2]:
# Sign-aware favorability: revenue under budget reads UNFAVORABLE
line = out['variance_detail']
r = line[(line['month']=='2025-09-01') & (line['account_id']=='REV_SUB')].iloc[0]
print(f'Subscription Revenue: actual ${r["actual"]:,.0f} vs budget ${r["budget"]:,.0f} -> favorable={r["favorable_ab"]}')

Subscription Revenue: actual $2,434,216 vs budget $2,539,250 -> favorable=False


In [3]:
# Top drivers for 2025-09 by absolute OI impact
from calculations import driver_decomposition as dd
dd.top_drivers(line, '2025-09-01', n=6)[['department_id','account_name','var_ab_amount','oi_impact_ab','favorable_ab']]

,department_id,account_name,var_ab_amount,oi_impact_ab,favorable_ab
0,CORP,Subscription Revenue,-105033.97,-105033.97,False
1,SM,Salaries,67060.83,-67060.83,False
2,RND,Contractors,62579.68,-62579.68,False
3,SM,Paid Marketing,43111.54,-43111.54,False
4,CORP,Hosting / Infrastructure,37208.93,-37208.93,False
5,CS,Salaries,-24631.00,24631.00,True


In [4]:
# Comp decomposition reconciles to the salary line
from calculations import driver_decomposition as dd
comp = dd.comp_decomposition(t['fact_headcount'])
print('max comp decomposition residual: $%.4f' % comp['decomp_residual'].abs().max())
comp[(comp['month']=='2025-09-01')][['department_id','salary_variance','hc_impact','rate_impact','decomp_residual']]

max comp decomposition residual: $0.0100


,department_id,salary_variance,hc_impact,rate_impact,decomp_residual
20,SM,67060.83,48333.33,18727.50,0.0
44,RND,591.88,-28000.00,28591.88,0.0
68,GA,3793.95,0.00,3793.95,0.0
92,CS,-24631.00,-28000.00,3369.00,0.0
